In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

## Load data files & Mouse trajectory preview

In [ ]:
from src.plotting import plot_trajectory
from src.data import find_red_eclipse_files, load_red_eclipse_mouse

game_files = find_red_eclipse_files()

for file_path in game_files[:5]:
    meta, mouse = load_red_eclipse_mouse(file_path)
    if mouse.empty:
        continue
    plot_trajectory(mouse, title=f"userId={meta['userId']}, gameId={meta['gameId']}")
    print(f"\nuserId={meta['userId']}, gameId={meta['gameId']}, events={len(mouse)}")


## Feature extraction


In [ ]:
from src.features import extract_features
from src.data import load_red_eclipse_mouse

game_rows = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    features = extract_features(mouse)

    if features is None:
        continue

    features.update({
        **meta,
        "is_bot": 0,
        "bot_type": "human",
    })
    game_rows.append(features)

games_df = pd.DataFrame(game_rows)
games_df.to_csv("../data/red_eclipse_features.csv", index=False)
print(games_df.head())


## Player identification


In [ ]:
from src.features import feature_cols

MIN_GAMES = 8
games_df_37 = games_df.groupby("userId").filter(lambda g: len(g) >= MIN_GAMES)

# games_df = all 45 players, games_df_37 = more than 8 games
select_model = games_df_37

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))


## Block-bootstrap synthetic bots


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED
from src.data import load_red_eclipse_mouse

N_BOT_GAMES = len(games_df)  # one bot game per human game

rng = np.random.default_rng(RNG_SEED)

# 1) Build segment pool + empirical motion stats from human games
segment_pool = []
re_human_mice = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    re_human_mice.append(mouse)
    segment_pool.extend(build_segments(mouse, rng=rng))

re_motion = collect_human_motion_samples(re_human_mice, rng=rng)
re_dt_samples = re_motion["dt_samples"]
re_dt_by_session = re_motion["dt_by_session"]
re_target_ms = median_trace_duration_ms(re_human_mice)
print(
    f"Segment pool: {len(segment_pool)} segments from {len(game_files)} games | "
    f"dt n={len(re_dt_samples)} sessions={len(re_dt_by_session)} median={np.median(re_dt_samples):.2f}ms | "
    f"step median={np.median(re_motion['step_samples']):.3f} | "
    f"target={re_target_ms/1000:.1f}s"
)

# 2) Generate synthetic bot games
N_PREVIEW = 5              # save first N trajectories for render graph
bot_rows = []
sample_bot_trajectories = []
for i in range(N_BOT_GAMES):
    bot_mouse = stitch_bot_game(
        segment_pool,
        dt_samples=re_dt_samples,
        dt_by_session=re_dt_by_session,
        target_duration_ms=re_target_ms,
        rng=rng,
    )
    if len(sample_bot_trajectories) < N_PREVIEW:
        sample_bot_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"bot_{i}",
        "source_file": f"synthetic_bot_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    bot_rows.append(feats)

bots_stitch_df = pd.DataFrame(bot_rows)
print(f"Generated {len(bots_stitch_df)} stitch bot games")
print(bots_stitch_df.head())



## Block-bootstrap synthetic bots trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_bot_trajectories):
    plot_trajectory(df, title=f"bot_{i}")
    print(f"\nbot_{i}, events={len(df)}")


## Smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_params_for_print,
)
from src.features import extract_features

median_events = int(games_df["n_events"].median())
N_SMOOTH_BOTS = len(games_df)

re_smooth_params = estimate_smooth_params(games_df, **re_motion)
print(f"RE smooth params: {smooth_params_for_print(re_smooth_params)}")

smooth_rows = []
sample_smooth_trajectories = []
for i in range(N_SMOOTH_BOTS):
    bot_mouse = generate_smooth_bot_game(
        n_events=median_events, seed=RNG_SEED + i, **re_smooth_params
    )
    if len(sample_smooth_trajectories) < N_PREVIEW:
        sample_smooth_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"smooth_{i}",
        "source_file": f"synthetic_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    smooth_rows.append(feats)

bots_smooth_df = pd.DataFrame(smooth_rows)
print(f"Generated {len(bots_smooth_df)} smooth bot games (n_events={median_events})")
print(bots_smooth_df.head())


## Smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_smooth_trajectories):
    plot_trajectory(df, title=f"smooth_{i}")
    print(f"\nsmooth_{i}, events={len(df)}")

## Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features

N_BEZIER_BOTS = len(games_df)

re_bezier_params = estimate_bezier_params(games_df, **re_motion)
print(f"RE bezier params: {bezier_params_for_print(re_bezier_params)}")

bezier_rows = []
sample_bezier_trajectories = []
for i in range(N_BEZIER_BOTS):
    bot_mouse = generate_bezier_bot_game(
        n_events=median_events, seed=RNG_SEED + 50 + i, **re_bezier_params
    )
    if len(sample_bezier_trajectories) < N_PREVIEW:
        sample_bezier_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -3,
        "gameId": f"bezier_{i}",
        "source_file": f"synthetic_bezier_{i}",
        "is_bot": 1,
        "bot_type": "bezier",
    })
    bezier_rows.append(feats)

bots_bezier_df = pd.DataFrame(bezier_rows)
print(f"Generated {len(bots_bezier_df)} bezier bot games (n_events={median_events})")
print(bots_bezier_df.head())



## Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_bezier_trajectories):
    plot_trajectory(df, title=f"bezier_{i}")
    print(f"\nbezier_{i}, events={len(df)}")



## RE Human vs Bot classification (GroupKFold + split-first + **windows**)

Window-level in-domain: same **split-first** `GroupKFold(5)` by `userId` as `main_split_first`, but each ~3 min trace is cut into non-overlapping **10s** windows (configurable via `WINDOW_MS`). Bots stay full-length, then windowed the same way.

Reports **window-level** mean±std and **session-level** (mean window P(bot)) for comparison with the session pipeline. Full-pool bots above stay for preview + cross-game.


In [ ]:
from src.evaluation import evaluate_split_first_group_kfold_windows
from src.features import feature_cols
from src.data import load_red_eclipse_mouse
from src.config import RNG_SEED, WINDOW_MS, WINDOW_MIN_EVENTS
from pathlib import Path

# Override here if you want a different window (defaults from src.config)
RE_WINDOW_MS = WINDOW_MS          # 10_000
RE_WINDOW_MIN_EVENTS = WINDOW_MIN_EVENTS  # 30

re_file_by_name = {Path(p).name: p for p in game_files}

def _re_mice_for(feat_df):
    return [load_red_eclipse_mouse(re_file_by_name[src])[1] for src in feat_df["source_file"]]

re_cv = evaluate_split_first_group_kfold_windows(
    games_df,
    groups=games_df["userId"].to_numpy(),
    mice_for_df=_re_mice_for,
    feature_cols=feature_cols,
    n_splits=5,
    random_state=RNG_SEED,
    rng_seed_base=RNG_SEED,
    round_deltas=True,
    name="RE",
    window_ms=RE_WINDOW_MS,
    min_events=RE_WINDOW_MIN_EVENTS,
)
re_cv_summary = re_cv["summary_df"]
re_cv_session_summary = re_cv["session_summary_df"]
re_cv_folds = re_cv["fold_df"]
print("\nRE fold table (window counts):")
print(
    re_cv_folds[
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nRE window-level summary:")
print(re_cv_summary.to_string(index=False))
if re_cv_session_summary is not None:
    print("\nRE session-mean-of-windows summary:")
    print(re_cv_session_summary.to_string(index=False))


## Load LoL dataset

In [ ]:
from src.data import parse_lol_keylogger, find_lol_keylogger_files

RE_TRAIN_N = None
LOL_FILE_N = None
LOL_WINDOW_MIN = 3
LOL_PARSE_MAX_EVENTS = None
N_LOL_BOTS = None
SKIP_FULL_LOL_TRAJECTORY = False

lol_keylogger_files = find_lol_keylogger_files()

sample_lol = parse_lol_keylogger(
    lol_keylogger_files[0],
    max_events=LOL_PARSE_MAX_EVENTS,
    max_minutes=LOL_WINDOW_MIN,
)
print(f"File: {lol_keylogger_files[0].name}")
print(f"Events: {len(sample_lol)}, duration: {sample_lol['time'].iloc[-1] / 60000:.1f} min")
print(sample_lol.head())


## Load LoL sessions & extract features

In [ ]:
from src.features import extract_features
from src.data import parse_lol_keylogger, find_lol_keylogger_files

lol_keylogger_files = find_lol_keylogger_files()

files_to_load = lol_keylogger_files[:LOL_FILE_N] if LOL_FILE_N else lol_keylogger_files
print(f"Loading {len(files_to_load)} LoL files")

lol_rows = []
for file_path in files_to_load:
    mouse = parse_lol_keylogger(
        file_path,
        max_events=LOL_PARSE_MAX_EVENTS,
        max_minutes=LOL_WINDOW_MIN,
    )
    if mouse is None:
        continue
    mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
    feats = extract_features(mouse)
    if feats is None:
        continue

    session_date = file_path.parent.name
    participant = file_path.stem.split("-")[1]
    feats.update({
        "userId": f"lol_{participant}",
        "gameId": f"{session_date}_p{participant}",
        "source_file": file_path.name,
        "session_date": session_date,
        "is_bot": 0,
        "bot_type": "human",
    })
    lol_rows.append(feats)

lol_games_df = pd.DataFrame(lol_rows)
print(f"Loaded {len(lol_games_df)} LoL human sessions (first {LOL_WINDOW_MIN} min each)")
print(lol_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")


## LoL trajectory preview

In [ ]:
from src.data import parse_lol_keylogger, find_lol_keylogger_files

lol_keylogger_files = find_lol_keylogger_files()

PREVIEW_MINUTES = LOL_WINDOW_MIN

preview_lol = parse_lol_keylogger(
    lol_keylogger_files[0],
    max_events=LOL_PARSE_MAX_EVENTS,
    max_minutes=PREVIEW_MINUTES,
)
preview_lol = preview_lol[preview_lol["time"] <= PREVIEW_MINUTES * 60 * 1000].copy()
preview_lol["trajectory_x"] = preview_lol["dx"].cumsum()
preview_lol["trajectory_y"] = preview_lol["dy"].cumsum()

print(f"Zoom: first {PREVIEW_MINUTES} min, {len(preview_lol)} events")

if SKIP_FULL_LOL_TRAJECTORY:
    plt.figure(figsize=(7, 5))
    plt.plot(preview_lol["trajectory_x"], preview_lol["trajectory_y"], linewidth=0.5, color="steelblue")
    plt.title(f"LoL — first {PREVIEW_MINUTES} min")
    plt.gca().invert_yaxis()
    plt.axis("equal")
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(preview_lol["trajectory_x"], preview_lol["trajectory_y"], linewidth=0.5, color="steelblue")
    axes[0].set_title(f"LoL — first {PREVIEW_MINUTES} min")
    axes[0].invert_yaxis()
    axes[0].set_aspect("equal")
    full_lol = parse_lol_keylogger(lol_keylogger_files[0])
    full_lol["trajectory_x"] = full_lol["dx"].cumsum()
    full_lol["trajectory_y"] = full_lol["dy"].cumsum()
    axes[1].plot(full_lol["trajectory_x"], full_lol["trajectory_y"], linewidth=0.1, color="steelblue", alpha=0.3)
    axes[1].set_title(f"LoL — full session ({len(full_lol)/1e6:.2f}M events)")
    axes[1].invert_yaxis()
    axes[1].set_aspect("equal")
    plt.tight_layout()
    plt.show()


## Cross-game transfer (RE train → LoL test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

re_human = games_df.head(RE_TRAIN_N) if RE_TRAIN_N else games_df
re_stitch = bots_stitch_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_stitch_df
re_smooth = bots_smooth_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_smooth_df
re_bezier = bots_bezier_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_bezier_df

print("=== RE model trained on STITCH bots ===")
re_model_stitch, re_acc_stitch = train_bot_detector(re_human, re_stitch, cross_game_feature_cols, name="stitch")
print()
print("=== RE model trained on SMOOTH bots ===")
re_model_smooth, re_acc_smooth = train_bot_detector(re_human, re_smooth, cross_game_feature_cols, name="smooth")
print()
print("=== RE model trained on BEZIER bots ===")
re_model_bezier, re_acc_bezier = train_bot_detector(re_human, re_bezier, cross_game_feature_cols, name="bezier")



## LoL data human identifier

In [ ]:
# test will LoL humans identified as bot
for name, model in [
    ("stitch", re_model_stitch),
    ("smooth", re_model_smooth),
    ("bezier", re_model_bezier),
]:
    pred = model.predict(lol_games_df[cross_game_feature_cols])
    print(f"[{name} model] LoL human false-positive rate: {pred.mean():.2%} ({pred.sum()}/{len(pred)} identified as bot)")



## LoL stitch bot generation

In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.data import parse_lol_keylogger
from src.features import extract_features

lol_segment_pool = []
lol_human_mice = []
for file_path in files_to_load:
    mouse = parse_lol_keylogger(
        file_path,
        max_events=LOL_PARSE_MAX_EVENTS,
        max_minutes=LOL_WINDOW_MIN,
    )
    if mouse is None:
        continue
    mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
    lol_human_mice.append(mouse)

lol_bot_rng = np.random.default_rng(RNG_SEED + 1)
for mouse in lol_human_mice:
    lol_segment_pool.extend(build_segments(mouse, rng=lol_bot_rng))

lol_motion = collect_human_motion_samples(lol_human_mice, rng=lol_bot_rng)
lol_dt_samples = lol_motion["dt_samples"]
lol_dt_by_session = lol_motion["dt_by_session"]
lol_target_ms = median_trace_duration_ms(lol_human_mice)
print(
    f"LoL dt n={len(lol_dt_samples)} sessions={len(lol_dt_by_session)} median={np.median(lol_dt_samples):.2f}ms | "
    f"step median={np.median(lol_motion['step_samples']):.3f} | "
    f"target={lol_target_ms/1000:.1f}s"
)

n_lol_bots = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)
lol_stitch_rows = []
sample_lol_stitch_trajectories = []

for i in range(n_lol_bots):
    bot_mouse = stitch_bot_game(
        lol_segment_pool,
        dt_samples=lol_dt_samples,
        dt_by_session=lol_dt_by_session,
        target_duration_ms=lol_target_ms,
        rng=lol_bot_rng,
    )
    if len(sample_lol_stitch_trajectories) < N_PREVIEW:
        sample_lol_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -1, "gameId": f"lol_stitch_{i}", "is_bot": 1, "bot_type": "stitch"})
    lol_stitch_rows.append(feats)

lol_bots_stitch_df = pd.DataFrame(lol_stitch_rows)
print(f"LoL stitch bots: {len(lol_bots_stitch_df)} (target {lol_target_ms/1000:.1f}s each)")



## LoL stitch bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_stitch_trajectories):
    print(f"lol_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"lol_stitch_{i}")

## LoL smooth bot generation

In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_params_for_print,
)

lol_median_events = int(lol_games_df["n_events"].median())
n_lol_smooth = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)

lol_smooth_params = estimate_smooth_params(lol_games_df, **lol_motion)
print(f"LoL smooth params: {smooth_params_for_print(lol_smooth_params)}")
print(f"(RE smooth params for comparison: {smooth_params_for_print(re_smooth_params)})")

lol_smooth_rows = []
sample_lol_smooth_trajectories = []
for i in range(n_lol_smooth):
    bot_mouse = generate_smooth_bot_game(
        n_events=lol_median_events, seed=RNG_SEED + 100 + i, **lol_smooth_params
    )
    if len(sample_lol_smooth_trajectories) < N_PREVIEW:
        sample_lol_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -2, "gameId": f"lol_smooth_{i}", "is_bot": 1, "bot_type": "smooth"})
    lol_smooth_rows.append(feats)

lol_bots_smooth_df = pd.DataFrame(lol_smooth_rows)
print(f"LoL smooth bots: {len(lol_bots_smooth_df)} (n_events={lol_median_events})")


## LoL smooth bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_smooth_trajectories):
    plot_trajectory(df, title=f"lol_smooth_{i}")
    print(f"lol_smooth_{i}, events={len(df)}")


## LoL Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)

lol_median_events = int(lol_games_df["n_events"].median())
n_lol_bezier = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)

lol_bezier_params = estimate_bezier_params(lol_games_df, **lol_motion)
print(f"LoL bezier params: {bezier_params_for_print(lol_bezier_params)}")
print(f"(RE bezier params for comparison: {bezier_params_for_print(re_bezier_params)})")

lol_bezier_rows = []
sample_lol_bezier_trajectories = []
for i in range(n_lol_bezier):
    bot_mouse = generate_bezier_bot_game(
        n_events=lol_median_events, seed=RNG_SEED + 150 + i, **lol_bezier_params
    )
    if len(sample_lol_bezier_trajectories) < N_PREVIEW:
        sample_lol_bezier_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -3, "gameId": f"lol_bezier_{i}", "is_bot": 1, "bot_type": "bezier"})
    lol_bezier_rows.append(feats)

lol_bots_bezier_df = pd.DataFrame(lol_bezier_rows)
print(f"LoL bezier bots: {len(lol_bots_bezier_df)} (n_events={lol_median_events})")



## LoL Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_bezier_trajectories):
    plot_trajectory(df, title=f"lol_bezier_{i}")
    print(f"lol_bezier_{i}, events={len(df)}")



## LoL in-domain (GroupKFold + split-first + **windows**)

Same windowing as RE (default 10s). Group by `userId`. With only ~7 users, folds are small — report **mean ± std**, not a single unlucky split.


In [ ]:
from src.evaluation import evaluate_split_first_group_kfold_windows
from src.features import feature_cols
from src.data import parse_lol_keylogger
from src.config import RNG_SEED, WINDOW_MS, WINDOW_MIN_EVENTS

LOL_SEG_WINDOW_MS = WINDOW_MS
LOL_SEG_MIN_EVENTS = WINDOW_MIN_EVENTS

lol_path_by_name = {p.name: p for p in files_to_load}

def _lol_mice_for(feat_df):
    mice = []
    for src in feat_df["source_file"]:
        mouse = parse_lol_keylogger(
            lol_path_by_name[src],
            max_events=LOL_PARSE_MAX_EVENTS,
            max_minutes=LOL_WINDOW_MIN,
        )
        mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
        mice.append(mouse)
    return mice

lol_cv = evaluate_split_first_group_kfold_windows(
    lol_games_df,
    groups=lol_games_df["userId"].to_numpy(),
    mice_for_df=_lol_mice_for,
    feature_cols=feature_cols,
    n_splits=5,
    random_state=RNG_SEED,
    rng_seed_base=RNG_SEED + 1,
    round_deltas=True,
    name="LoL",
    window_ms=LOL_SEG_WINDOW_MS,
    min_events=LOL_SEG_MIN_EVENTS,
)
lol_cv_summary = lol_cv["summary_df"]
lol_cv_session_summary = lol_cv["session_summary_df"]
lol_cv_folds = lol_cv["fold_df"]
print("\nLoL fold table (window counts):")
print(
    lol_cv_folds[
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nLoL window-level summary:")
print(lol_cv_summary.to_string(index=False))
if lol_cv_session_summary is not None:
    print("\nLoL session-mean-of-windows summary:")
    print(lol_cv_session_summary.to_string(index=False))


## Feature scale comparison (RE vs LoL)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs LoL human vs LoL bots):")
compare = pd.DataFrame({
    "RE_human": games_df[cols].median(),
    "LoL_human": lol_games_df[cols].median(),
    "LoL_stitch": lol_bots_stitch_df[cols].median(),
    "LoL_smooth": lol_bots_smooth_df[cols].median(),
    "LoL_bezier": lol_bots_bezier_df[cols].median(),
}).round(3)
print(compare)



## Zero-shot diagnose (raw features, RE → LoL)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, lol_games_df, lol_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, lol_games_df, lol_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, lol_games_df, lol_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)



## Scale-invariant train (RE)

Train RF on unitless ratio features (same RE subset as raw cross-game models).



In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

# Same RE training subset as raw cross-game models when RE_TRAIN_N is set
re_si_human = to_scale_invariant(re_human)
re_si_stitch = to_scale_invariant(re_stitch)
re_si_smooth = to_scale_invariant(re_smooth)
re_si_bezier = to_scale_invariant(re_bezier)

print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)




## Zero-shot diagnose (scale-invariant features, RE → LoL)

Same `diagnose_zero_shot` suite as raw: AUC, P(bot), @0.5, human-calibrated threshold, sweep.



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

lol_si_human = to_scale_invariant(lol_games_df)
lol_si_stitch = to_scale_invariant(lol_bots_stitch_df)
lol_si_smooth = to_scale_invariant(lol_bots_smooth_df)
lol_si_bezier = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)




## RE→LoL Bézier inversion diagnostic

Why raw AUC≈0.1 (scores flipped): compare feature medians RE vs LoL, and which high-importance features reverse human↔bot order across games.


In [ ]:
import pandas as pd
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS

raw_cols = cross_game_feature_cols

# --- (1) Feature medians: RE human / RE bezier / LoL human / LoL bezier ---
print("=== Raw 7-feature medians ===")
raw_med = pd.DataFrame({
    "RE_human": games_df[raw_cols].median(),
    "RE_bezier": bots_bezier_df[raw_cols].median(),
    "LoL_human": lol_games_df[raw_cols].median(),
    "LoL_bezier": lol_bots_bezier_df[raw_cols].median(),
}).round(4)
# signed gap human - bot (same game); flip if RE and LoL gaps have opposite sign
raw_med["gap_RE"] = (raw_med["RE_human"] - raw_med["RE_bezier"]).round(4)
raw_med["gap_LoL"] = (raw_med["LoL_human"] - raw_med["LoL_bezier"]).round(4)
raw_med["sign_flip"] = (raw_med["gap_RE"] * raw_med["gap_LoL"]) < 0
print(raw_med)
print()

re_sf = to_scale_invariant(games_df)
re_bz_sf = to_scale_invariant(bots_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant 4-feature medians ===")
sf_med = pd.DataFrame({
    "RE_human": re_sf[SCALE_INVARIANT_COLS].median(),
    "RE_bezier": re_bz_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_bezier": lol_bz_sf[SCALE_INVARIANT_COLS].median(),
}).round(4)
sf_med["gap_RE"] = (sf_med["RE_human"] - sf_med["RE_bezier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL_bezier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_RE"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

# --- (2) Feature importance of the RE-trained bezier detectors ---
print("=== RE bezier model importance (raw 7-feat, used in RE→LoL raw) ===")
imp_raw = pd.Series(
    re_model_bezier.feature_importances_, index=raw_cols
).sort_values(ascending=False)
print(imp_raw.round(4))
print()

print("=== RE bezier model importance (scale-invariant, used in RE→LoL SF) ===")
imp_sf = pd.Series(
    m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

# --- Cross-read: high importance ∩ sign flip ---
print("=== Suspects: importance rank + sign_flip ===")
print("Raw:")
for feat, imp in imp_raw.items():
    flip = bool(raw_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={raw_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={raw_med.loc[feat, 'gap_LoL']:+.4f}")
print("Scale-invariant:")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={sf_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")




## Feature importance (scale-invariant models)



In [ ]:
import pandas as pd
from src.features import SCALE_INVARIANT_COLS

imp = pd.Series(m_si_stitch.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant stitch model) ===")
print(imp)
print()
imp2 = pd.Series(m_si_smooth.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant smooth model) ===")
print(imp2)
print()
imp3 = pd.Series(m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant bezier model) ===")
print(imp3)




## CSGO load: eye_vector → (dx, dy, time)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import CSGO_DATA_ROOT
from src.data_csgo import (
    check_axis_convention,
    eye_vectors_to_mouse_df,
    validate_real_roundtrip,
)

USECOLS = ["time", "eyeVectorX", "eyeVectorY", "eyeVectorZ"]

def find_gameflt_files(root=CSGO_DATA_ROOT):
    files = sorted(Path(root).rglob("gameFlt.csv"))
    print(f"Found {len(files)} gameFlt.csv under {root}")
    return files

def load_eye_csv(path):
    df = pd.read_csv(path, usecols=USECOLS)
    finite = np.isfinite(df[["eyeVectorX", "eyeVectorY", "eyeVectorZ"]]).all(axis=1)
    return df.loc[finite].reset_index(drop=True)

def convert_one(path):
    flt = load_eye_csv(path)
    mouse_df, meta = eye_vectors_to_mouse_df(
        flt["time"], flt["eyeVectorX"], flt["eyeVectorY"], flt["eyeVectorZ"]
    )
    return mouse_df, meta, flt

gameflt_paths = find_gameflt_files()
assert gameflt_paths, f"No gameFlt.csv under {CSGO_DATA_ROOT}"

first_path = gameflt_paths[0]
print(f"\n=== First-file checks: {first_path} ===")

mouse0, meta0, flt0 = convert_one(first_path)
print("convert meta:", meta0)
print(mouse0.head())

axis_ok = check_axis_convention(flt0["eyeVectorY"])
rt = validate_real_roundtrip(
    flt0["eyeVectorX"], flt0["eyeVectorY"], flt0["eyeVectorZ"]
)

if not (axis_ok and rt["ok"]):
    raise RuntimeError(
        "First-file checks failed — stop before converting all files. "
        f"axis_ok={axis_ok}, roundtrip_ok={rt['ok']}"
    )

print("\nFirst file PASSED. Converting all gameFlt.csv ...")

csgo_mouse = {}
rows = []
for i, path in enumerate(gameflt_paths, 1):
    # path like .../S001/P3/gameFlt.csv
    participant = path.parent.name          # P3
    session = path.parent.parent.name       # S001
    key = (session, participant)
    try:
        mouse_df, meta, _ = convert_one(path)
    except Exception as e:
        print(f"  SKIP {key}: {e}")
        continue
    csgo_mouse[key] = mouse_df
    rows.append({
        "session": session,
        "participant": participant,
        "n_out": meta["n_out"],
        "teleport_frac": meta["teleport_frac"],
        "path": str(path),
    })
    if i % 50 == 0 or i == len(gameflt_paths):
        print(f"  converted {i}/{len(gameflt_paths)}")

csgo_convert_summary = pd.DataFrame(rows)
print(f"\nDone: {len(csgo_mouse)} traces")
print(csgo_convert_summary.head())
print(
    "n_out median:", csgo_convert_summary["n_out"].median(),
    "| teleport_frac median:", f"{csgo_convert_summary['teleport_frac'].median():.2%}",
)

## CSGO sessions & extract features


In [ ]:
from pathlib import Path

from src.features import extract_features
from src.config import CSGO_DATA_ROOT, CSGO_WINDOW_MIN
from src.data_csgo import window_mouse_round_alive

print(
    f"Extracting features from {len(csgo_mouse)} CSGO traces "
    f"(Round2+alive, {CSGO_WINDOW_MIN} min)"
)

csgo_mouse_win = {}
csgo_rows = []
csgo_window_meta = []
n_skip = 0

for (session, participant), mouse in csgo_mouse.items():
    session_dir = Path(CSGO_DATA_ROOT) / session / participant
    win, meta = window_mouse_round_alive(
        mouse, session_dir, window_min=CSGO_WINDOW_MIN
    )
    csgo_window_meta.append({"session": session, "participant": participant, **meta})
    if not meta.get("ok") or win is None:
        n_skip += 1
        continue

    feats = extract_features(win)
    if feats is None:
        n_skip += 1
        continue

    csgo_mouse_win[(session, participant)] = win
    feats.update({
        "userId": f"csgo_{session}_{participant}",
        "gameId": f"{session}_{participant}",
        "session": session,
        "participant": participant,
        "is_bot": 0,
        "bot_type": "human",
        "round_n": meta["round_n"],
        "alive_frac_in_window": meta["alive_frac_in_window"],
    })
    csgo_rows.append(feats)

csgo_games_df = pd.DataFrame(csgo_rows)
csgo_window_meta_df = pd.DataFrame(csgo_window_meta)

print(
    f"Loaded {len(csgo_games_df)} CSGO human sessions "
    f"(Round2+alive, skipped {n_skip})"
)
print(
    "alive_frac_in_window median:",
    f"{csgo_games_df['alive_frac_in_window'].median():.1%}",
)
print(csgo_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"CSGO — median n_events: {csgo_games_df['n_events'].median():.0f}")
print(csgo_games_df.head())


## CSGO trajectory preview


In [ ]:
from src.plotting import plot_trajectory
from src.config import CSGO_WINDOW_MIN

preview_keys = list(csgo_mouse_win.keys())[:5]

for session, participant in preview_keys:
    mouse = csgo_mouse_win[(session, participant)]
    plot_trajectory(
        mouse,
        title=f"CSGO {session}/{participant} (Round2+alive, {CSGO_WINDOW_MIN} min)",
    )
    print(f"\n{session}/{participant}, events={len(mouse)}")


## CSGO stitch bot generation


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED

N_PREVIEW = 5
N_CSGO_BOTS = None  # None = one bot per CSGO human session

csgo_bot_rng = np.random.default_rng(RNG_SEED + 2)
csgo_segment_pool = []
for mouse in csgo_mouse_win.values():
    csgo_segment_pool.extend(build_segments(mouse, rng=csgo_bot_rng))

csgo_motion = collect_human_motion_samples(csgo_mouse_win.values(), rng=csgo_bot_rng)
csgo_dt_samples = csgo_motion["dt_samples"]
csgo_dt_by_session = csgo_motion["dt_by_session"]
csgo_target_ms = median_trace_duration_ms(csgo_mouse_win.values())
print(
    f"CSGO segment pool: {len(csgo_segment_pool)} segments "
    f"from {len(csgo_mouse_win)} Round2+alive traces | "
    f"dt n={len(csgo_dt_samples)} sessions={len(csgo_dt_by_session)} median={np.median(csgo_dt_samples):.2f}ms | "
    f"step median={np.median(csgo_motion['step_samples']):.3f} | "
    f"target={csgo_target_ms/1000:.1f}s"
)

n_csgo_bots = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)
csgo_stitch_rows = []
sample_csgo_stitch_trajectories = []

for i in range(n_csgo_bots):
    bot_mouse = stitch_bot_game(
        csgo_segment_pool,
        dt_samples=csgo_dt_samples,
        dt_by_session=csgo_dt_by_session,
        target_duration_ms=csgo_target_ms,
        rng=csgo_bot_rng,
    )
    if len(sample_csgo_stitch_trajectories) < N_PREVIEW:
        sample_csgo_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"csgo_stitch_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    csgo_stitch_rows.append(feats)

csgo_bots_stitch_df = pd.DataFrame(csgo_stitch_rows)
print(f"CSGO stitch bots: {len(csgo_bots_stitch_df)} (target {csgo_target_ms/1000:.1f}s each)")
print(csgo_bots_stitch_df.head())



## CSGO stitch bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_stitch_trajectories):
    print(f"csgo_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"csgo_stitch_{i}")


## CSGO smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_params_for_print,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())
n_csgo_smooth = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)

csgo_smooth_params = estimate_smooth_params(csgo_games_df, **csgo_motion)
print(f"CSGO smooth params: {smooth_params_for_print(csgo_smooth_params)}")

csgo_smooth_rows = []
sample_csgo_smooth_trajectories = []
for i in range(n_csgo_smooth):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_smooth_bot_game(
        n_events=csgo_median_events,
        seed=RNG_SEED + 200 + i,
        round_deltas=False,
        **csgo_smooth_params,
    )
    if len(sample_csgo_smooth_trajectories) < N_PREVIEW:
        sample_csgo_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"csgo_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    csgo_smooth_rows.append(feats)

csgo_bots_smooth_df = pd.DataFrame(csgo_smooth_rows)
print(f"CSGO smooth bots: {len(csgo_bots_smooth_df)} (n_events={csgo_median_events})")
print(csgo_bots_smooth_df.head())


## CSGO smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_smooth_trajectories):
    plot_trajectory(df, title=f"csgo_smooth_{i}")
    print(f"csgo_smooth_{i}, events={len(df)}")


## CSGO Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())
n_csgo_bezier = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)

csgo_bezier_params = estimate_bezier_params(csgo_games_df, **csgo_motion)
print(f"CSGO bezier params: {bezier_params_for_print(csgo_bezier_params)}")

csgo_bezier_rows = []
sample_csgo_bezier_trajectories = []
for i in range(n_csgo_bezier):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_bezier_bot_game(
        n_events=csgo_median_events,
        seed=RNG_SEED + 250 + i,
        round_deltas=False,
        **csgo_bezier_params,
    )
    if len(sample_csgo_bezier_trajectories) < N_PREVIEW:
        sample_csgo_bezier_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -3,
        "gameId": f"csgo_bezier_{i}",
        "is_bot": 1,
        "bot_type": "bezier",
    })
    csgo_bezier_rows.append(feats)

csgo_bots_bezier_df = pd.DataFrame(csgo_bezier_rows)
print(f"CSGO bezier bots: {len(csgo_bots_bezier_df)} (n_events={csgo_median_events})")
print(csgo_bots_bezier_df.head())



## CSGO Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_bezier_trajectories):
    plot_trajectory(df, title=f"csgo_bezier_{i}")
    print(f"csgo_bezier_{i}, events={len(df)}")



## CSGO in-domain (GroupKFold + split-first + **windows**)

Same windowing as RE (default 10s). Group by `participant` (person-level; same person never crosses train/test). With 4 participants, `GroupKFold` uses **4 folds** (capped). Report mean ± std.


In [ ]:
from src.evaluation import evaluate_split_first_group_kfold_windows
from src.features import feature_cols
from src.config import RNG_SEED, WINDOW_MS, WINDOW_MIN_EVENTS

CSGO_SEG_WINDOW_MS = WINDOW_MS
CSGO_SEG_MIN_EVENTS = WINDOW_MIN_EVENTS

def _csgo_mice_for(feat_df):
    return [
        csgo_mouse_win[(row.session, row.participant)]
        for row in feat_df.itertuples(index=False)
    ]

csgo_cv = evaluate_split_first_group_kfold_windows(
    csgo_games_df,
    groups=csgo_games_df["participant"].to_numpy(),
    mice_for_df=_csgo_mice_for,
    feature_cols=feature_cols,
    n_splits=5,  # capped to n_participants (=4)
    random_state=RNG_SEED,
    rng_seed_base=RNG_SEED + 2,
    round_deltas=False,
    name="CSGO",
    window_ms=CSGO_SEG_WINDOW_MS,
    min_events=CSGO_SEG_MIN_EVENTS,
)
csgo_cv_summary = csgo_cv["summary_df"]
csgo_cv_session_summary = csgo_cv["session_summary_df"]
csgo_cv_folds = csgo_cv["fold_df"]
print("\nCSGO fold table (window counts):")
print(
    csgo_cv_folds[
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nCSGO window-level summary:")
print(csgo_cv_summary.to_string(index=False))
if csgo_cv_session_summary is not None:
    print("\nCSGO session-mean-of-windows summary:")
    print(csgo_cv_session_summary.to_string(index=False))


## Zero-shot diagnose (raw features, RE → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, csgo_games_df, csgo_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, csgo_games_df, csgo_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, csgo_games_df, csgo_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)



## Scale-invariant train (RE) — for CSGO transfer


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

re_si_human = to_scale_invariant(re_human)
re_si_stitch = to_scale_invariant(re_stitch)
re_si_smooth = to_scale_invariant(re_smooth)
re_si_bezier = to_scale_invariant(re_bezier)

print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)




## Zero-shot diagnose (scale-invariant features, RE → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)

print("=== Scale-invariant: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)




## Feature scale comparison (RE vs CSGO)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs CSGO human vs CSGO bots):")
compare_re_csgo = pd.DataFrame({
    "RE_human": games_df[cols].median(),
    "CSGO_human": csgo_games_df[cols].median(),
    "CSGO_stitch": csgo_bots_stitch_df[cols].median(),
    "CSGO_smooth": csgo_bots_smooth_df[cols].median(),
    "CSGO_bezier": csgo_bots_bezier_df[cols].median(),
}).round(3)
print(compare_re_csgo)



## Cross-game transfer (CSGO train → RE test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

print("=== CSGO model trained on STITCH bots (7 cross-game features) ===")
csgo_x_model_stitch, csgo_x_acc_stitch = train_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, cross_game_feature_cols, name="CSGO stitch"
)
print()
print("=== CSGO model trained on SMOOTH bots (7 cross-game features) ===")
csgo_x_model_smooth, csgo_x_acc_smooth = train_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, cross_game_feature_cols, name="CSGO smooth"
)
print()
print("=== CSGO model trained on BEZIER bots (7 cross-game features) ===")
csgo_x_model_bezier, csgo_x_acc_bezier = train_bot_detector(
    csgo_games_df, csgo_bots_bezier_df, cross_game_feature_cols, name="CSGO bezier"
)



## Zero-shot diagnose (raw features, CSGO → RE)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", csgo_x_model_stitch, games_df, bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", csgo_x_model_smooth, games_df, bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", csgo_x_model_bezier, games_df, bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)



## Scale-invariant train (CSGO) — for RE transfer



In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_si_stitch_tr = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth_tr = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier_tr = to_scale_invariant(csgo_bots_bezier_df)

print("=== Train on CSGO (scale-invariant features) ===")
m_csgo_si_stitch, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_stitch_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_csgo_si_smooth, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_smooth_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_csgo_si_bezier, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_bezier_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant bezier",
    show_feature_importance=False,
)




## Zero-shot diagnose (scale-invariant features, CSGO → RE)



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

re_si_human = to_scale_invariant(games_df)
re_si_stitch = to_scale_invariant(bots_stitch_df)
re_si_smooth = to_scale_invariant(bots_smooth_df)
re_si_bezier = to_scale_invariant(bots_bezier_df)

print("=== Scale-invariant: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)




## Feature scale comparison (CSGO vs RE)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (CSGO human vs RE human vs RE bots):")
compare_csgo_re = pd.DataFrame({
    "CSGO_human": csgo_games_df[cols].median(),
    "RE_human": games_df[cols].median(),
    "RE_stitch": bots_stitch_df[cols].median(),
    "RE_smooth": bots_smooth_df[cols].median(),
    "RE_bezier": bots_bezier_df[cols].median(),
}).round(3)
print(compare_csgo_re)

